# Fine-tuning a Pretrained Model (Tinh chỉnh mô hình)
## Phân loại văn bản xem một câu có phải nói về thảm họa thật hay không

# Bước 1: Install Libraries

In [1]:
pip install transformers datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


# Bước 2: Load Dataset

In [ ]:
from datasets import load_dataset

# 1 Load dữ liệu
dataset = load_dataset('csv', data_files={'train': 'train.csv'})

# 2 Đổi tên cột 'target' thành 'label' để tương thích thư viện
dataset = dataset.rename_column("target", "label")

# 3 Xóa cột thừa chỉ giữ lại 'text' và 'label'
dataset = dataset.remove_columns(["id", "keyword", "location"])

# 4 Chia dữ liệu: 80% Train - 20% Test 
# seed=42 để kết quả chia giống nhau mỗi lần chạy
dataset_split = dataset['train'].train_test_split(test_size=0.2, seed=42)

print(dataset_split)


Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 6090
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1523
    })
})


# Bước 3 & 4: Load Tokenizer & Preprocess

In [ ]:
#  Tải Tokenizer & Tiền xử lý dữ liệu
from transformers import AutoTokenizer

# 1 Tải Tokenizer của DistilBERT: chuyển chữ thường, không phân biệt hoa thường
model_name = "distilbert-base-uncased"
#  Chuyển đổi toàn bộ văn bản trong dataset thành các chuỗi số
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2 Hàm xử lý (Tokenize được cắt ngắn hoặc thêm padding cho bằng nhau
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# 3 Áp dụng hàm trên vào toàn bộ dataset
tokenized_datasets = dataset_split.map(tokenize_function, batched=True)

print("Xử lý xong dữ liệu!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/6090 [00:00<?, ? examples/s]

Map:   0%|          | 0/1523 [00:00<?, ? examples/s]

Xử lý xong dữ liệu!


# Bước 5: Define Training Arguments

In [ ]:
# Thiết lập tham số huấn luyện
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="test_trainer",          # Nơi lưu model
    eval_strategy="epoch",              # Kiểm tra điểm sau mỗi vòng học 
    learning_rate=2e-5,                 # Tốc độ học 2e-5 
    per_device_train_batch_size=16,     # Số lượng câu học cùng lúc (tùy RAM máy)
    per_device_eval_batch_size=16,      # Số lượng câu kiểm tra cùng lúc
    num_train_epochs=2,                 # Học 2 vòng
    weight_decay=0.01,                  # Tăng cường để tránh overfit
)

# Bước 6: Create Trainer & Finetune

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer
import evaluate
import numpy as np

# 1 Tải Model (num_labels=2 vì chỉ có 0 và 1)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 2 Chuẩn bị hàm tính điểm Accuracy
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 3 Tạo Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

# 4 BẮT ĐẦU TRAIN
trainer.train()

# 5 LƯU MODEL
save_path = "./my_fine_tuned_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f" Đã lưu model và tokenizer vào: {save_path}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

wandb: Enter your choice:

wandb: Enter your choice:

wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.398247,0.828628
2,0.410200,0.407328,0.838477


 Đã lưu model và tokenizer vào: ./my_fine_tuned_model


* Nhận xét: 
    * Mô hình học qua 2 epoch loss giảm xuống còn 0.41
    * Accuracy: 83.8%

# Bước 7: Evaluate

In [6]:
# Chạy đánh giá
results = trainer.evaluate()

print("Kết quả đánh giá:")
print(results)


Kết quả đánh giá:
{'eval_loss': 0.4073276221752167, 'eval_accuracy': 0.8384766907419566, 'eval_runtime': 24.0378, 'eval_samples_per_second': 63.359, 'eval_steps_per_second': 3.994, 'epoch': 2.0}


* Nhan xet: eval_loss: 0.407 (sai số thấp), eval_accuracy: 0.838 (độ chính xác khá cao)

# Bước 8: Thử nghiệm thực tế (Inference)

In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

save_path = "./my_fine_tuned_model"

print("Đang tải model")
try:
    # Ưu tiên load từ model đã lưu trên ổ cứng
    loaded_model = AutoModelForSequenceClassification.from_pretrained(save_path)
    loaded_tokenizer = AutoTokenizer.from_pretrained(save_path)
    print(" Đã load thành công model từ ổ cứng")
except Exception as e:
    print(f" Không tìm thấy model đã lưu: {e}")
    print(" Đang thử sử dụng model vừa train xong ")
    try:
        loaded_model = trainer.model
        loaded_tokenizer = tokenizer
        print(" Đang sử dụng model vừa train xong.")
    except NameError:
        print(" LỖI: Không tìm thấy model nào cả. Hãy chạy bước Train trước!")

# Hàm dự đoán
def predict(text):
    inputs = loaded_tokenizer(text, return_tensors="pt", padding=True, truncation=True)

    # Chuyển sang CPU để để test cho đơn giản (hoặc GPU nếu model đang ở GPU)
    device = loaded_model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = loaded_model(**inputs).logits

    pred_id = logits.argmax().item()
    return "THẢM HỌA (Disaster) " if pred_id == 1 else "Bình thường (Not Disaster) "

# Test thử
test_texts = [
    "There is a huge fire in the forest!", # Có một đám cháy lớn trong rừng!
    "I am watching a movie with my friends.", # Tôi đang xem phim với bạn bè.
    "The earthquake destroyed many buildings.",     # Trận động đất đã phá hủy nhiều tòa nhà.
    "Just finished a 5k run, feeling great!", # Vừa chạy xong 5km, cảm thấy rất tuyệt!

]

print("\n--- KẾT QUẢ DỰ ĐOÁN ---")
for t in test_texts:
    print(f"Câu: '{t}'\n -> Dự đoán: {predict(t)}\n")

Đang tải model
 Đã load thành công model từ ổ cứng

--- KẾT QUẢ DỰ ĐOÁN ---
Câu: 'There is a huge fire in the forest!'
 -> Dự đoán: THẢM HỌA (Disaster) 

Câu: 'I am watching a movie with my friends.'
 -> Dự đoán: Bình thường (Not Disaster) 

Câu: 'The earthquake destroyed many buildings.'
 -> Dự đoán: THẢM HỌA (Disaster) 

Câu: 'Just finished a 5k run, feeling great!'
 -> Dự đoán: Bình thường (Not Disaster) 

